
# CC-Perturb-seq sequencing and guide-capture QC

This notebook calculates the manuscript QC quantities:

1. **Average transcriptome UMIs per cell**
2. **Average transcriptome reads per cell**
3. **Average guide UMIs per guide per cell**

It uses the integrated CC-Perturb-seq AnnData path extracted from the two supplied notebooks and automatically inventories candidate count/read/guide fields before calculating anything.

## Paths extracted from the supplied notebooks

| Resource | Extracted path | Used here? |
|---|---|---|
| Integrated AnnData | `/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/2.preprocessing_tonys_pipeline/3.integrated_days/3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad` | **Yes; primary input** |
| cNMF MuData | `/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/adata/cNMF_50_0_2.h5mu` | Optional cell-index cross-check only |
| cNMF usage table | `/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50.usages.k_50.dt_0_2.consensus.txt` | Not needed for sequencing QC |
| Guide metadata | `300genes_guide_metadata_v43.tsv` | Optional; relative path in the supplied notebook |

**Important:** neither supplied notebook contained an explicit path to a GEX sequencing-read metrics file. Therefore, average transcriptome reads per cell can be calculated immediately only if a per-cell read-count field is stored in the AnnData object. Otherwise, set `GEX_LIBRARY_METRICS_CSV` below to a library-level sequencing summary table.


In [ ]:

from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

class cfg:
    # ------------------------------------------------------------------
    # Paths extracted from the attached notebooks
    # ------------------------------------------------------------------
    ADATA_H5AD = Path(
        '/oak/stanford/groups/engreitz/Users/opushkar/cc_perturb/'
        '2.preprocessing_tonys_pipeline/3.integrated_days/'
        '3.all_days_concat_pc30_all_genes_sceptre_guides.h5ad'
    )

    H5MU_CNMf = Path(
        '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/'
        '030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/'
        'adata/cNMF_50_0_2.h5mu'
    )

    CNMF_USAGE_TXT = Path(
        '/oak/stanford/groups/engreitz/Users/ymo/IGVF_ccperturbseq/Result/'
        '030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50/'
        '030526_100k_cells_100iter_allHVG_torch_halsvar_batch_e7_50.'
        'usages.k_50.dt_0_2.consensus.txt'
    )

    # Relative path in the carrier-neighborhood notebook. This is optional
    # for the three averages, but useful for mapping/validating guide columns.
    GUIDE_METADATA_TSV = Path('300genes_guide_metadata_v43.tsv')

    OUTDIR = Path('CC_Perturbseq_sequencing_guide_QC_outputs')

    # ------------------------------------------------------------------
    # Optional manual overrides. Leave as None for auto-detection.
    # ------------------------------------------------------------------
    GEX_UMI_OBS_COL = None
    GEX_READ_OBS_COL = None
    GUIDE_TOTAL_UMI_OBS_COL = None
    N_ASSIGNED_GUIDES_OBS_COL = None
    GUIDE_UMI_OBSM_KEY = None

    GUIDE_ASSIGNMENT_OBSM_KEY = 'guide_assignment'
    GUIDE_NAMES_UNS_KEY = 'guide_names'

    # Optional counts-layer fallback for transcriptome UMIs.
    # Use only if this is an integer/raw-count layer; do not point to
    # normalized/log-transformed X.
    GEX_COUNTS_LAYER = None

    # Optional library-level read summary when per-cell reads are not stored.
    # Expected as one row per library/sample. It can contain either:
    #   (a) total reads + post-QC cell count, or
    #   (b) a precomputed mean reads per cell.
    GEX_LIBRARY_METRICS_CSV = None
    LIBRARY_ID_COL = None
    LIBRARY_TOTAL_READS_COL = None
    LIBRARY_CELL_COUNT_COL = None
    LIBRARY_MEAN_READS_PER_CELL_COL = None

    # Candidate sample/day columns for stratified summaries.
    DAY_COL_CANDIDATES = ['day', 'day_clean', 'timepoint', 'time_point']
    SAMPLE_COL_CANDIDATES = [
        'biological_sample', 'sample', 'library_id', 'library',
        'batch', 'replicate', 'sample_id'
    ]

cfg.OUTDIR.mkdir(parents=True, exist_ok=True)
print('Output directory:', cfg.OUTDIR.resolve())
print('AnnData:', cfg.ADATA_H5AD)



## 1. Load AnnData in backed mode

The notebook first uses `backed='r'` so that the large expression matrix is not loaded into memory. The three requested averages should normally be obtained from per-cell QC fields or guide-count matrices stored in the AnnData object.


In [ ]:

if not cfg.ADATA_H5AD.exists():
    raise FileNotFoundError(
        f'AnnData file not found: {cfg.ADATA_H5AD}\n'
        'Update cfg.ADATA_H5AD if the Sherlock path has moved.'
    )

adata = sc.read_h5ad(cfg.ADATA_H5AD, backed='r')
print(adata)
print(f'Post-QC cells in AnnData: {adata.n_obs:,}')
print(f'Genes/features in AnnData: {adata.n_vars:,}')



## 2. Inventory candidate QC fields

This step is deliberately verbose. It prevents accidental use of a normalized matrix or a Boolean guide-assignment matrix as a UMI-count matrix.


In [ ]:

def summarize_obs_columns(adata):
    patterns = re.compile(r'(count|umi|read|guide|grna|rna|sample|batch|rep|day)', re.I)
    rows = []
    for col in adata.obs.columns:
        if patterns.search(str(col)):
            s = adata.obs[col]
            rows.append({
                'field': str(col),
                'dtype': str(s.dtype),
                'n_non_missing': int(s.notna().sum()),
                'n_unique': int(s.nunique(dropna=True)),
                'example_values': ', '.join(map(str, s.dropna().astype(str).head(4).tolist())),
            })
    return pd.DataFrame(rows).sort_values('field') if rows else pd.DataFrame()

obs_inventory = summarize_obs_columns(adata)
display(obs_inventory)

other_inventory = pd.DataFrame({
    'container': ['obsm', 'layers', 'uns'],
    'keys': [
        ', '.join(map(str, adata.obsm_keys())),
        ', '.join(map(str, adata.layers.keys())),
        ', '.join(map(str, adata.uns_keys())),
    ],
})
display(other_inventory)

obs_inventory.to_csv(cfg.OUTDIR / 'candidate_obs_fields.csv', index=False)
other_inventory.to_csv(cfg.OUTDIR / 'anndata_container_key_inventory.csv', index=False)


## 3. Helper functions for conservative field detection and summary

In [ ]:

GEX_UMI_EXACT = [
    'total_counts', 'n_counts', 'nCount_RNA', 'total_umis',
    'transcriptome_umi_counts', 'gex_umi_counts', 'rna_umi_counts'
]
GEX_READ_EXACT = [
    'total_reads', 'mapped_reads', 'gex_reads', 'transcriptome_reads',
    'read_count', 'reads_per_cell', 'n_reads'
]
GUIDE_TOTAL_UMI_EXACT = [
    'guide_umi_counts', 'total_guide_umis', 'guide_umis',
    'grna_umi_counts', 'gRNA_umi_counts', 'total_grna_umis'
]
N_ASSIGNED_GUIDES_EXACT = [
    'guides_per_cell', 'n_assigned_guides', 'n_total_guides',
    'number_of_guides', 'guide_count'
]
GUIDE_UMI_OBSM_EXACT = [
    'guide_umi_counts', 'guide_counts', 'guide_umis',
    'grna_umi_counts', 'gRNA_umi_counts', 'guide_capture_counts'
]


def numeric_series(adata, col):
    if col is None or col not in adata.obs.columns:
        return None
    s = pd.to_numeric(adata.obs[col], errors='coerce')
    if s.notna().sum() == 0:
        return None
    return s.astype(float)


def resolve_obs_col(explicit, candidates, label):
    if explicit is not None:
        if explicit not in adata.obs.columns:
            raise KeyError(f'{label}: configured column {explicit!r} not found in adata.obs')
        if numeric_series(adata, explicit) is None:
            raise TypeError(f'{label}: configured column {explicit!r} is not numeric')
        print(f'{label}: using configured adata.obs[{explicit!r}]')
        return explicit

    for col in candidates:
        if col in adata.obs.columns and numeric_series(adata, col) is not None:
            print(f'{label}: auto-detected adata.obs[{col!r}]')
            return col

    print(f'{label}: no exact candidate detected')
    return None


def resolve_first_existing(candidates, existing):
    for key in candidates:
        if key in existing:
            return key
    return None


def find_first_col(candidates):
    for c in candidates:
        if c in adata.obs.columns:
            return c
    return None


def summarize_numeric(x, label, source):
    x = pd.to_numeric(pd.Series(x), errors='coerce').replace([np.inf, -np.inf], np.nan).dropna()
    if len(x) == 0:
        return None
    return {
        'metric': label,
        'source': source,
        'n_observations': int(len(x)),
        'mean': float(x.mean()),
        'median': float(x.median()),
        'standard_deviation': float(x.std(ddof=1)) if len(x) > 1 else np.nan,
        'minimum': float(x.min()),
        'maximum': float(x.max()),
    }


def matrix_is_boolean(x):
    if sparse.issparse(x):
        return x.dtype == bool
    arr = np.asarray(x)
    return arr.dtype == bool


def safe_matrix_sum(x):
    if sparse.issparse(x):
        return float(x.sum())
    return float(np.asarray(x).sum())


def safe_matrix_nnz_positive(x):
    if sparse.issparse(x):
        # Copying .data is small relative to densifying the matrix.
        return int(np.count_nonzero(np.asarray(x.data) > 0))
    return int(np.count_nonzero(np.asarray(x) > 0))


def row_sums(x):
    if sparse.issparse(x):
        return np.asarray(x.sum(axis=1)).ravel().astype(float)
    return np.asarray(x).sum(axis=1).astype(float)


def col_sums(x):
    if sparse.issparse(x):
        return np.asarray(x.sum(axis=0)).ravel().astype(float)
    return np.asarray(x).sum(axis=0).astype(float)


def positive_counts_by_col(x):
    if sparse.issparse(x):
        return np.asarray((x > 0).sum(axis=0)).ravel().astype(float)
    return (np.asarray(x) > 0).sum(axis=0).astype(float)


def row_sum_counts_layer(adata, layer, chunk_size=2000):
    if layer is None:
        raise ValueError('A raw integer counts layer must be configured explicitly.')
    if layer not in adata.layers:
        raise KeyError(f'Layer {layer!r} not found. Available layers: {list(adata.layers.keys())}')
    totals = np.empty(adata.n_obs, dtype=float)
    for start in range(0, adata.n_obs, chunk_size):
        stop = min(start + chunk_size, adata.n_obs)
        block = adata.layers[layer][start:stop, :]
        totals[start:stop] = row_sums(block)
    return totals



## 4. Average transcriptome UMIs per cell

Preferred source: a post-QC per-cell field such as `total_counts`. A counts-layer fallback is used only if explicitly configured because the integrated expression matrix may be normalized or log-transformed.


In [ ]:

summary_rows = []
metric_status = []
cell_metrics = pd.DataFrame(index=adata.obs_names.copy())

umi_col = resolve_obs_col(cfg.GEX_UMI_OBS_COL, GEX_UMI_EXACT, 'Transcriptome UMI count')

if umi_col is not None:
    gex_umis = numeric_series(adata, umi_col)
    cell_metrics['transcriptome_umis'] = gex_umis.to_numpy()
    summary_rows.append(summarize_numeric(
        gex_umis,
        'Transcriptome UMIs per cell',
        f'adata.obs[{umi_col!r}]'
    ))
    metric_status.append({
        'requested_metric': 'Average transcriptome UMIs per cell',
        'status': 'CALCULATED',
        'source': f'adata.obs[{umi_col!r}]',
        'note': 'Arithmetic mean across post-QC cells in the integrated AnnData object.'
    })
elif cfg.GEX_COUNTS_LAYER is not None:
    gex_umis = pd.Series(
        row_sum_counts_layer(adata, cfg.GEX_COUNTS_LAYER),
        index=adata.obs_names,
        name='transcriptome_umis'
    )
    cell_metrics['transcriptome_umis'] = gex_umis.to_numpy()
    summary_rows.append(summarize_numeric(
        gex_umis,
        'Transcriptome UMIs per cell',
        f'row sum of adata.layers[{cfg.GEX_COUNTS_LAYER!r}]'
    ))
    metric_status.append({
        'requested_metric': 'Average transcriptome UMIs per cell',
        'status': 'CALCULATED_FROM_EXPLICIT_COUNTS_LAYER',
        'source': f'adata.layers[{cfg.GEX_COUNTS_LAYER!r}]',
        'note': 'Confirm that the configured layer contains integer raw UMI counts.'
    })
else:
    metric_status.append({
        'requested_metric': 'Average transcriptome UMIs per cell',
        'status': 'NOT CALCULATED',
        'source': '',
        'note': 'No trusted per-cell UMI column was detected and no raw counts layer was configured.'
    })

pd.DataFrame([r for r in summary_rows if r is not None])



## 5. Average transcriptome reads per cell

Reads are different from UMIs. The integrated AnnData may not retain read counts. This cell tries, in order:

1. a per-cell read-count column in `adata.obs`;
2. a user-supplied library metrics table containing total reads and cell counts; or
3. a user-supplied library metrics table containing mean reads per cell.

If none is available, the result is explicitly marked unavailable rather than substituting UMI counts.


In [ ]:

read_col = resolve_obs_col(cfg.GEX_READ_OBS_COL, GEX_READ_EXACT, 'Transcriptome read count')

if read_col is not None:
    gex_reads = numeric_series(adata, read_col)
    cell_metrics['transcriptome_reads'] = gex_reads.to_numpy()
    summary_rows.append(summarize_numeric(
        gex_reads,
        'Transcriptome reads per cell',
        f'adata.obs[{read_col!r}]'
    ))
    metric_status.append({
        'requested_metric': 'Average transcriptome reads per cell',
        'status': 'CALCULATED',
        'source': f'adata.obs[{read_col!r}]',
        'note': 'Arithmetic mean across post-QC cells.'
    })

elif cfg.GEX_LIBRARY_METRICS_CSV is not None:
    metrics_path = Path(cfg.GEX_LIBRARY_METRICS_CSV)
    if not metrics_path.exists():
        raise FileNotFoundError(metrics_path)
    lib = pd.read_csv(metrics_path)
    print('Library metrics columns:', lib.columns.tolist())

    total_reads_col = cfg.LIBRARY_TOTAL_READS_COL
    cell_count_col = cfg.LIBRARY_CELL_COUNT_COL
    mean_reads_col = cfg.LIBRARY_MEAN_READS_PER_CELL_COL

    if mean_reads_col is not None:
        values = pd.to_numeric(lib[mean_reads_col], errors='coerce')
        if cell_count_col is not None:
            weights = pd.to_numeric(lib[cell_count_col], errors='coerce')
            ok = values.notna() & weights.notna() & (weights > 0)
            overall_mean_reads = float(np.average(values[ok], weights=weights[ok]))
            source_note = f'cell-count-weighted {mean_reads_col!r} from {metrics_path}'
        else:
            overall_mean_reads = float(values.mean())
            source_note = f'unweighted mean of {mean_reads_col!r} from {metrics_path}'
    elif total_reads_col is not None and cell_count_col is not None:
        total_reads = pd.to_numeric(lib[total_reads_col], errors='coerce').sum()
        total_cells = pd.to_numeric(lib[cell_count_col], errors='coerce').sum()
        overall_mean_reads = float(total_reads / total_cells)
        source_note = f'sum({total_reads_col}) / sum({cell_count_col}) from {metrics_path}'
    else:
        raise ValueError(
            'Configure either LIBRARY_MEAN_READS_PER_CELL_COL, or both '
            'LIBRARY_TOTAL_READS_COL and LIBRARY_CELL_COUNT_COL.'
        )

    summary_rows.append({
        'metric': 'Transcriptome reads per cell',
        'source': source_note,
        'n_observations': int(len(lib)),
        'mean': overall_mean_reads,
        'median': np.nan,
        'standard_deviation': np.nan,
        'minimum': np.nan,
        'maximum': np.nan,
    })
    metric_status.append({
        'requested_metric': 'Average transcriptome reads per cell',
        'status': 'CALCULATED_FROM_LIBRARY_METRICS',
        'source': source_note,
        'note': 'Check that the denominator matches the manuscript definition of cells.'
    })

else:
    metric_status.append({
        'requested_metric': 'Average transcriptome reads per cell',
        'status': 'NEEDS READ-METRICS FILE',
        'source': '',
        'note': (
            'The supplied notebooks contain no GEX read-metrics path. Set '
            'GEX_LIBRARY_METRICS_CSV or GEX_READ_OBS_COL. Do not substitute total_counts, '
            'which represents UMIs rather than reads.'
        )
    })

pd.DataFrame([r for r in summary_rows if r is not None])



## 6. Average guide UMIs per guide per cell

Because cells can carry multiple guides, the notebook reports several transparent definitions:

- **Guide UMIs per cell:** total guide UMIs in each cell, averaged over all post-QC cells.
- **Global guide UMIs per assigned guide-cell pair:** total guide UMIs divided by the total number of assigned guide-cell pairs. This is the preferred interpretation of “UMIs per guide per cell.”
- **Mean within-cell UMIs per assigned guide:** for each guide-positive cell, total guide UMIs divided by the number of assigned guides, then averaged across cells.
- If a full numeric cell-by-guide UMI matrix exists, the notebook additionally reports the mean UMI count per nonzero guide-cell pair and unweighted per-guide detection summaries.

The Boolean `guide_assignment` matrix is never treated as a UMI-count matrix.


In [ ]:

# Resolve assignment matrix, which is allowed to be Boolean.
assignment = None
if cfg.GUIDE_ASSIGNMENT_OBSM_KEY in adata.obsm:
    assignment = adata.obsm[cfg.GUIDE_ASSIGNMENT_OBSM_KEY]
    if sparse.issparse(assignment):
        assignment = assignment.tocsr()
    print(
        f'Guide assignment matrix: adata.obsm[{cfg.GUIDE_ASSIGNMENT_OBSM_KEY!r}], '
        f'shape={assignment.shape}, dtype={assignment.dtype}'
    )
else:
    print('No guide_assignment matrix found.')

# Resolve a numeric cell-by-guide UMI matrix if present.
if cfg.GUIDE_UMI_OBSM_KEY is not None:
    if cfg.GUIDE_UMI_OBSM_KEY not in adata.obsm:
        raise KeyError(f'Configured guide UMI matrix key not found: {cfg.GUIDE_UMI_OBSM_KEY}')
    guide_umi_key = cfg.GUIDE_UMI_OBSM_KEY
else:
    guide_umi_key = resolve_first_existing(GUIDE_UMI_OBSM_EXACT, adata.obsm_keys())

# Do not accept the assignment matrix itself as a numeric UMI matrix.
if guide_umi_key == cfg.GUIDE_ASSIGNMENT_OBSM_KEY:
    guide_umi_key = None

# Resolve total guide UMI and guide-count obs fields.
guide_total_umi_col = resolve_obs_col(
    cfg.GUIDE_TOTAL_UMI_OBS_COL,
    GUIDE_TOTAL_UMI_EXACT,
    'Total guide UMI count per cell'
)
assigned_guides_col = resolve_obs_col(
    cfg.N_ASSIGNED_GUIDES_OBS_COL,
    N_ASSIGNED_GUIDES_EXACT,
    'Number of assigned guides per cell'
)

guide_matrix = None
if guide_umi_key is not None:
    candidate = adata.obsm[guide_umi_key]
    if matrix_is_boolean(candidate):
        print(f'Ignoring Boolean adata.obsm[{guide_umi_key!r}] as a UMI-count matrix.')
        guide_umi_key = None
    else:
        guide_matrix = candidate.tocsr() if sparse.issparse(candidate) else np.asarray(candidate)
        print(
            f'Guide UMI matrix: adata.obsm[{guide_umi_key!r}], '
            f'shape={guide_matrix.shape}, dtype={guide_matrix.dtype}'
        )

# Get per-cell total guide UMIs.
if guide_matrix is not None:
    total_guide_umis_per_cell = row_sums(guide_matrix)
    guide_umi_source = f'row sums of adata.obsm[{guide_umi_key!r}]'
elif guide_total_umi_col is not None:
    total_guide_umis_per_cell = numeric_series(adata, guide_total_umi_col).to_numpy()
    guide_umi_source = f'adata.obs[{guide_total_umi_col!r}]'
else:
    total_guide_umis_per_cell = None
    guide_umi_source = ''

# Get number of assigned guides per cell.
if assigned_guides_col is not None:
    n_assigned_guides_per_cell = numeric_series(adata, assigned_guides_col).to_numpy()
    assigned_source = f'adata.obs[{assigned_guides_col!r}]'
elif assignment is not None:
    n_assigned_guides_per_cell = row_sums(assignment)
    assigned_source = f'row sums of adata.obsm[{cfg.GUIDE_ASSIGNMENT_OBSM_KEY!r}]'
elif guide_matrix is not None:
    if sparse.issparse(guide_matrix):
        n_assigned_guides_per_cell = np.asarray((guide_matrix > 0).sum(axis=1)).ravel().astype(float)
    else:
        n_assigned_guides_per_cell = (guide_matrix > 0).sum(axis=1).astype(float)
    assigned_source = f'nonzero entries of adata.obsm[{guide_umi_key!r}]'
else:
    n_assigned_guides_per_cell = None
    assigned_source = ''

if total_guide_umis_per_cell is not None:
    cell_metrics['guide_umis_total'] = total_guide_umis_per_cell
    summary_rows.append(summarize_numeric(
        total_guide_umis_per_cell,
        'Guide UMIs per cell, all post-QC cells',
        guide_umi_source
    ))

if n_assigned_guides_per_cell is not None:
    cell_metrics['n_assigned_guides'] = n_assigned_guides_per_cell
    summary_rows.append(summarize_numeric(
        n_assigned_guides_per_cell,
        'Assigned guides per cell, all post-QC cells',
        assigned_source
    ))

if total_guide_umis_per_cell is not None and n_assigned_guides_per_cell is not None:
    valid = (
        np.isfinite(total_guide_umis_per_cell)
        & np.isfinite(n_assigned_guides_per_cell)
        & (n_assigned_guides_per_cell > 0)
    )
    global_ratio = float(
        np.nansum(total_guide_umis_per_cell[valid])
        / np.nansum(n_assigned_guides_per_cell[valid])
    )
    per_cell_ratio = total_guide_umis_per_cell[valid] / n_assigned_guides_per_cell[valid]
    cell_metrics.loc[cell_metrics.index[valid], 'guide_umis_per_assigned_guide_in_cell'] = per_cell_ratio

    summary_rows.append({
        'metric': 'Guide UMIs per assigned guide-cell pair, ratio of totals',
        'source': f'{guide_umi_source}; denominator from {assigned_source}',
        'n_observations': int(valid.sum()),
        'mean': global_ratio,
        'median': np.nan,
        'standard_deviation': np.nan,
        'minimum': np.nan,
        'maximum': np.nan,
    })
    summary_rows.append(summarize_numeric(
        per_cell_ratio,
        'Mean within-cell guide UMIs per assigned guide',
        f'{guide_umi_source} / {assigned_source}; guide-positive cells only'
    ))

    metric_status.append({
        'requested_metric': 'Average guide UMIs per guide/cell',
        'status': 'CALCULATED',
        'source': f'{guide_umi_source}; {assigned_source}',
        'note': (
            'Preferred manuscript number is the ratio-of-totals: total guide UMIs divided '
            'by total assigned guide-cell pairs. The notebook also reports the mean per-cell ratio.'
        )
    })
else:
    metric_status.append({
        'requested_metric': 'Average guide UMIs per guide/cell',
        'status': 'NOT CALCULATED',
        'source': '',
        'note': (
            'Need either a numeric cell-by-guide UMI matrix, or both a per-cell total guide-UMI '
            'column and the number/Boolean assignment of guides per cell.'
        )
    })

# Additional exact metrics if full numeric guide UMI matrix is present.
per_guide_summary = pd.DataFrame()
if guide_matrix is not None:
    total_guide_umis = safe_matrix_sum(guide_matrix)
    nonzero_pairs = safe_matrix_nnz_positive(guide_matrix)
    mean_per_nonzero_pair = total_guide_umis / nonzero_pairs if nonzero_pairs else np.nan

    summary_rows.append({
        'metric': 'Guide UMIs per nonzero guide-cell pair',
        'source': f'adata.obsm[{guide_umi_key!r}]',
        'n_observations': int(nonzero_pairs),
        'mean': float(mean_per_nonzero_pair),
        'median': np.nan,
        'standard_deviation': np.nan,
        'minimum': np.nan,
        'maximum': np.nan,
    })

    names = None
    if cfg.GUIDE_NAMES_UNS_KEY in adata.uns:
        names = list(map(str, adata.uns[cfg.GUIDE_NAMES_UNS_KEY]))
    if names is None or len(names) != guide_matrix.shape[1]:
        names = [f'guide_{i}' for i in range(guide_matrix.shape[1])]

    sums = col_sums(guide_matrix)
    npos = positive_counts_by_col(guide_matrix)
    per_guide_summary = pd.DataFrame({
        'guide_id': names,
        'total_umis': sums,
        'n_cells_with_positive_umi': npos.astype(int),
        'mean_umis_per_positive_cell': np.divide(
            sums, npos, out=np.full_like(sums, np.nan, dtype=float), where=npos > 0
        ),
    })
    per_guide_summary.to_csv(cfg.OUTDIR / 'guide_umi_summary_by_guide.csv', index=False)

summary_df = pd.DataFrame([r for r in summary_rows if r is not None])
display(summary_df)


## 7. Stratify available per-cell metrics by day and sample/replicate

In [ ]:

day_col = find_first_col(cfg.DAY_COL_CANDIDATES)
sample_col = find_first_col(cfg.SAMPLE_COL_CANDIDATES)

if day_col is not None:
    cell_metrics['day'] = adata.obs[day_col].astype(str).to_numpy()
if sample_col is not None:
    cell_metrics['sample'] = adata.obs[sample_col].astype(str).to_numpy()

metric_cols = [
    c for c in [
        'transcriptome_umis', 'transcriptome_reads',
        'guide_umis_total', 'n_assigned_guides',
        'guide_umis_per_assigned_guide_in_cell'
    ] if c in cell_metrics.columns
]


def grouped_summary(df, group_col, metrics):
    if group_col not in df.columns or not metrics:
        return pd.DataFrame()
    parts = []
    for metric in metrics:
        g = df.groupby(group_col, dropna=False)[metric]
        out = g.agg(['count', 'mean', 'median', 'std', 'min', 'max']).reset_index()
        out.insert(1, 'metric', metric)
        parts.append(out)
    return pd.concat(parts, ignore_index=True)

by_day = grouped_summary(cell_metrics, 'day', metric_cols)
by_sample = grouped_summary(cell_metrics, 'sample', metric_cols)

display(Markdown(f'**Detected day column:** `{day_col}`'))
display(by_day)
display(Markdown(f'**Detected sample/replicate column:** `{sample_col}`'))
display(by_sample)

if not by_day.empty:
    by_day.to_csv(cfg.OUTDIR / 'sequencing_guide_qc_by_day.csv', index=False)
if not by_sample.empty:
    by_sample.to_csv(cfg.OUTDIR / 'sequencing_guide_qc_by_sample.csv', index=False)


## 8. Final status, manuscript-ready values, and exports

In [ ]:

status_df = pd.DataFrame(metric_status)
summary_df = pd.DataFrame([r for r in summary_rows if r is not None])

display(status_df)
display(summary_df)

status_df.to_csv(cfg.OUTDIR / 'requested_metric_status.csv', index=False)
summary_df.to_csv(cfg.OUTDIR / 'CC_Perturbseq_sequencing_guide_QC_summary.csv', index=False)

# Extract one recommended number for each manuscript placeholder.
def mean_for_metric(metric_name):
    hit = summary_df.loc[summary_df['metric'] == metric_name, 'mean']
    return float(hit.iloc[0]) if len(hit) else None

recommended = {
    'post_QC_cells': int(adata.n_obs),
    'average_transcriptome_UMIs_per_cell': mean_for_metric('Transcriptome UMIs per cell'),
    'average_transcriptome_reads_per_cell': mean_for_metric('Transcriptome reads per cell'),
    'average_guide_UMIs_per_assigned_guide_cell_pair': mean_for_metric(
        'Guide UMIs per assigned guide-cell pair, ratio of totals'
    ),
    'mean_within_cell_guide_UMIs_per_assigned_guide': mean_for_metric(
        'Mean within-cell guide UMIs per assigned guide'
    ),
    'average_total_guide_UMIs_per_cell': mean_for_metric(
        'Guide UMIs per cell, all post-QC cells'
    ),
}

with open(cfg.OUTDIR / 'manuscript_QC_values.json', 'w') as f:
    json.dump(recommended, f, indent=2)

print(json.dumps(recommended, indent=2))

lines = ['CC-Perturb-seq sequencing and guide QC', '']
for row in metric_status:
    lines.append(f"- {row['requested_metric']}: {row['status']}")
    if row['source']:
        lines.append(f"  Source: {row['source']}")
    if row['note']:
        lines.append(f"  Note: {row['note']}")
lines.append('')
lines.append('Recommended numerical values:')
for key, value in recommended.items():
    lines.append(f'- {key}: {value}')

(cfg.OUTDIR / 'QC_report.txt').write_text('\n'.join(lines))
print('\n'.join(lines))



## 9. How to report the guide metric

If both are available, use the following wording in the manuscript:

> Cells contained an average of **X transcriptome UMIs**, **Y transcriptome reads**, and **Z guide UMIs per assigned guide-cell pair**.

The ratio-of-totals guide metric is preferred because it weights every detected/assigned guide-cell pair equally. The mean of the within-cell ratios is also exported because it weights each guide-positive cell equally and can differ in a high-MOI experiment.

If the read metric remains unavailable, provide a path to either:

- a per-cell read-count table; or
- a library-level table containing total GEX reads and the matching post-QC cell count for each library.


In [ ]:

# Close the backed AnnData handle.
try:
    adata.file.close()
except Exception:
    pass
print('Finished. Outputs are in:', cfg.OUTDIR.resolve())
